In [1]:
# create_ieeg_rest_trials
"""
Trim iEEG .npz files to trial windows defined in rest_trials.csv.

Output: derivatives/dbs/trials/{dataset_key}/{subject}/{trial}.npz
Each trial file contains all channels for that subject/condition.
"""

import numpy as np
import pandas as pd
from pathlib import Path
import sys
import logging
import json
from dataclasses import dataclass
from typing import Dict, List, Optional

# =============================================================================
# CONFIGURATION
# =============================================================================

IEEG_BASE = Path('/mnt/movement/users/jaizor/xtra/derivatives/dbs')
TRIAL_CSV = Path('/mnt/movement/users/jaizor/xtra/notebooks/EEG/rest/__/rest_trials.csv')
OUTPUT_DIR = Path('/mnt/movement/users/jaizor/xtra/derivatives/dbs/trials')

@dataclass
class DatasetConfig:
    key: str
    ieeeg_dir: Path
    csv_path: Path
    trial_filter: Optional[List[int]] = None

# Only DBSOFF for now (matching your EEG script)
DATASETS = [
    DatasetConfig(
        key='rest_off',
        ieeeg_dir=IEEG_BASE,
        csv_path=TRIAL_CSV,
        trial_filter=None
    ),
]

# Subjects that have iEEG data (from our extraction)
IEEG_SUBJECTS = {
    'sub-01': 'RCS', 'sub-02': 'RCS', 'sub-03': 'Percept', 'sub-04': 'Percept',
    'sub-05': 'Percept', 'sub-06': 'Percept', 'sub-07': 'Percept',
    'sub-08': 'RCS', 'sub-09': 'Percept', 'sub-10': 'Percept',
    'sub-11': 'Percept', 'sub-12': 'RCS', 'sub-13': 'RCS', 'sub-14': 'RCS',
}

# =============================================================================
# LOGGING
# =============================================================================

def setup_logging(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger('ieeg_trials')
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    fh = logging.FileHandler(log_path, mode='w', encoding='utf-8')
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s'))
    
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter('%(levelname)-8s | %(message)s'))
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# =============================================================================
# HELPERS
# =============================================================================

def load_channel_npz(filepath: Path) -> tuple:
    """Load .npz file, return (data_array, metadata_dict, error_str)."""
    try:
        npz = np.load(filepath, allow_pickle=True)
        data = npz['data']
        meta = npz['metadata'].item() if npz['metadata'].ndim == 0 else npz['metadata']
        return data, meta, None
    except Exception as e:
        return None, None, str(e)

def crop_to_trial(data: np.ndarray, sfreq: float, t_min: float, t_max: float) -> np.ndarray:
    """Crop 1D time series to [t_min, t_max] seconds."""
    start_sample = int(np.floor(t_min * sfreq))
    end_sample = int(np.ceil(t_max * sfreq))
    
    # Bounds check
    if start_sample >= len(data):
        return np.array([])
    if end_sample > len(data):
        end_sample = len(data)
    
    return data[start_sample:end_sample]

def get_subject_channels(subject_dir: Path, dbs_state: str, task: str) -> Dict[str, Path]:
    """Find all channel .npz files for a subject/condition."""
    condition_dir = subject_dir / dbs_state / task
    if not condition_dir.exists():
        return {}
    
    channels = {}
    for f in condition_dir.glob('*.npz'):
        # Extract channel name from filename (handle both naming conventions)
        ch_name = f.stem
        if '_' in ch_name:
            # Percept: GPi-L_RHF_OFF_Left → GPi-L
            ch_name = ch_name.split('_')[0]
        channels[ch_name] = f
    return channels

def save_trial_npz(trial_data: Dict[str, np.ndarray], meta: Dict, output_path: Path):
    """Save trial with all channels as single .npz."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Stack channels into dict for saving
    save_dict = {ch: data.astype(np.float32) for ch, data in trial_data.items()}
    save_dict['trial_metadata'] = meta
    
    np.savez_compressed(output_path, **save_dict, allow_pickle=True)

# =============================================================================
# MAIN PIPELINE
# =============================================================================

def run_trial_extraction(config: DatasetConfig, logger: logging.Logger) -> int:
    """Extract iEEG trials for one dataset config."""
    
    logger.info(f"\n{'='*70}")
    logger.info(f"STARTING IEEG TRIAL EXTRACTION: {config.key}")
    logger.info(f"Source: {config.ieeeg_dir}")
    logger.info(f"Output: {OUTPUT_DIR / config.key}")
    logger.info(f"{'='*70}")
    
    if not config.csv_path.exists():
        logger.error(f"CSV not found: {config.csv_path}")
        return 0
    
    trials_df = pd.read_csv(config.csv_path)
    logger.info(f"Loaded {len(trials_df)} trials from {config.csv_path.name}")
    
    saved_count = 0
    output_root = OUTPUT_DIR / config.key
    
    # Process each subject that has iEEG data
    for sub_id in IEEG_SUBJECTS:
        device = IEEG_SUBJECTS[sub_id]
        subject_dir = config.ieeeg_dir / sub_id
        
        if not subject_dir.exists():
            logger.warning(f"⚠️  No iEEG data for {sub_id}")
            continue
        
        # Get trials for this subject
        sub_trials = trials_df[trials_df['subject'] == sub_id]
        if config.trial_filter:
            sub_trials = sub_trials[sub_trials['trial_number'].isin(config.trial_filter)]
        
        if sub_trials.empty:
            continue
        
        logger.info(f"\nProcessing {sub_id} ({device}, {len(sub_trials)} trials)...")
        
        # Load one file to get sfreq (all files for subject should match)
        sample_file = next(subject_dir.rglob('*.npz'), None)
        if sample_file is None:
            logger.warning(f"  ⊘ No .npz files found for {sub_id}")
            continue
        
        _, sample_meta, err = load_channel_npz(sample_file)
        if err:
            logger.error(f"  ✗ Could not read sample file: {err}")
            continue
        sfreq = sample_meta.get('sfreq', 250)
        
        # Process each trial
        for _, row in sub_trials.iterrows():
            trial_name = row['trial_name']
            t_min = row['start_time']
            t_max = row['end_time']
            
            # Get all channels for this subject/condition
            channels = get_subject_channels(subject_dir, 'DBSOFF', 'rest')
            if not channels:
                logger.warning(f"  ⊘ No DBSOFF/rest channels for {sub_id}")
                continue
            
            # Crop each channel to trial window
            trial_data = {}
            for ch_name, ch_path in channels.items():
                data, meta, err = load_channel_npz(ch_path)
                if err:
                    logger.debug(f"    ⊘ {ch_name}: Could not load: {err}")
                    continue
                
                cropped = crop_to_trial(data, sfreq, t_min, t_max)
                if len(cropped) == 0:
                    logger.debug(f"    ⊘ {ch_name}: Empty after cropping")
                    continue
                
                trial_data[ch_name] = cropped
            
            if not trial_data:
                logger.warning(f"  ⊘ Trial {row['trial_number']} ({trial_name}): No valid data")
                continue
            
            # Build trial metadata
            trial_meta = {
                'subject_id': sub_id,
                'device': device,
                'trial_number': int(row['trial_number']),
                'trial_name': trial_name,
                't_min': float(t_min),
                't_max': float(t_max),
                'duration_sec': float(t_max - t_min),
                'sfreq': float(sfreq),
                'channels': list(trial_data.keys()),
                'n_samples_per_channel': {ch: len(d) for ch, d in trial_data.items()},
            }
            
            # Save
            fname = f"{config.key}_{sub_id}_{trial_name}_ieeg.npz"
            out_path = output_root / sub_id / fname
            
            try:
                save_trial_npz(trial_data, trial_meta, out_path)
                logger.info(f"  ✓ {fname}: {len(trial_data)} channels, {trial_meta['duration_sec']:.1f}s")
                saved_count += 1
            except Exception as e:
                logger.error(f"  ✗ Save failed for {fname}: {e}")
    
    logger.info(f"\n✅ Finished {config.key}: {saved_count} trials saved")
    return saved_count

# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    logger = setup_logging(OUTPUT_DIR / 'ieeg_trial_extraction.log')
    
    logger.info("="*70)
    logger.info("IEEG REST TRIAL EXTRACTION")
    logger.info(f"Started: {pd.Timestamp.now().isoformat()}")
    logger.info("="*70)
    
    total_saved = 0
    summary = {}
    
    try:
        for cfg in DATASETS:
            count = run_trial_extraction(cfg, logger)
            summary[cfg.key] = count
            total_saved += count
        
        logger.info("\n" + "="*70)
        logger.info("EXTRACTION COMPLETE — SUMMARY")
        logger.info("="*70)
        for ds, count in summary.items():
            logger.info(f"{ds:15} : {count:4d} trials")
        logger.info(f"{'TOTAL':15} : {total_saved:4d} trials")
        logger.info(f"Output: {OUTPUT_DIR.resolve()}")
        logger.info("="*70)
        
    except KeyboardInterrupt:
        logger.warning("Interrupted by user")
    except Exception as e:
        logger.critical(f"Pipeline crashed: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

INFO     | ======================================================================
INFO     | IEEG REST TRIAL EXTRACTION
INFO     | Started: 2026-02-18T00:22:02.776093
INFO     | ======================================================================
INFO     | 
INFO     | STARTING IEEG TRIAL EXTRACTION: rest_off
INFO     | Source: /mnt/movement/users/jaizor/xtra/derivatives/dbs
INFO     | Output: /mnt/movement/users/jaizor/xtra/derivatives/dbs/trials/rest_off
INFO     | ======================================================================
INFO     | Loaded 48 trials from rest_trials.csv
INFO     | 
Processing sub-01 (RCS, 4 trials)...
INFO     |   ✓ rest_off_sub-01_c_ieeg.npz: 6 channels, 60.0s
INFO     |   ✓ rest_off_sub-01_o_ieeg.npz: 6 channels, 60.0s
INFO     |   ✓ rest_off_sub-01_l_ieeg.npz: 6 channels, 60.0s
INFO     |   ✓ rest_off_sub-01_r_ieeg.npz: 6 channels, 60.0s
INFO     | 
Processing sub-02 (RCS, 4 trials)...
WARNING  |   ⊘ No DBSOFF/rest channels for sub-02
WARNING  |   ⊘